# FFAI Conversation Management

This notebook demonstrates FFAI's conversation management capabilities:

1. **Client-level conversation history** -- the raw message list sent to the LLM
2. **Hot-swapping clients** -- switch between providers mid-conversation
3. **Clearing conversation** -- reset client history while retaining FFAI analytics
4. **Injecting messages** -- add user or assistant messages directly into history
5. **Cloning clients** -- create independent conversation branches
6. **Client history vs FFAI history** -- understanding the two layers

In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="coroutine.*was never awaited", category=RuntimeWarning)

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / "pyproject.toml").is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv  # noqa: E402

load_dotenv()

from ffai import FFAI  # noqa: E402
from ffai.Clients import FFLiteLLMClient  # noqa: E402

mistral_key = os.getenv("MISTRAL_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

mistral_client = FFLiteLLMClient(
    model_string="mistral/mistral-small-latest",
    api_key=mistral_key,
    temperature=0,
    max_tokens=80,
)
openai_client = FFLiteLLMClient(
    model_string="openai/gpt-4o-mini",
    api_key=openai_key,
    temperature=0,
    max_tokens=80,
)

ffai = FFAI(client=mistral_client)

print(f"Project root: {_project_root}")
print(f"Mistral client model: {mistral_client.model}")
print(f"OpenAI client model:  {openai_client.model}")
print(f"Active client:        {ffai.client.model}")
print("Ready")

11:39:08 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


11:39:09 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Project root: /home/aq/Documents/Source/ffai-standalone
Mistral client model: mistral-small-latest
OpenAI client model:  gpt-4o-mini
Active client:        mistral-small-latest
Ready


<div class="page-break"></div>

---

## 1. Client-Level Conversation History

Each FFAI client maintains a `conversation_history` list of `{role, content}`
message dicts. Every `generate_response` call appends the user prompt and
assistant response to this list, creating a rolling multi-turn conversation
that gets sent to the LLM on each call.

In [2]:
r1 = ffai.generate_response(
    prompt="My name is Alice. Remember it.",
    prompt_name="intro",
)
print(f"Response: {r1.response}")
print(f"\nClient history has {len(ffai.get_client_conversation_history())} messages")
for msg in ffai.get_client_conversation_history():
    role = msg.get("role", "?")
    content = msg.get("content", "")
    preview = content[:60] if isinstance(content, str) else str(content)[:60]
    print(f"  [{role:9s}] {preview}")

Response: Got it, Alice! I'll remember your name. How can I assist you today?

Client history has 2 messages
  [user     ] My name is Alice. Remember it.
  [assistant] Got it, Alice! I'll remember your name. How can I assist you


In [3]:
r2 = ffai.generate_response(
    prompt="What is my name?",
    prompt_name="recall",
)
print(f"Response: {r2.response}")
print(f"\nClient history now has {len(ffai.get_client_conversation_history())} messages")
for msg in ffai.get_client_conversation_history():
    role = msg.get("role", "?")
    content = msg.get("content", "")
    preview = content[:60] if isinstance(content, str) else str(content)[:60]
    print(f"  [{role:9s}] {preview}")

Response: Your name is Alice!

Client history now has 4 messages
  [user     ] My name is Alice. Remember it.
  [assistant] Got it, Alice! I'll remember your name. How can I assist you
  [user     ] What is my name?
  [assistant] Your name is Alice!


<div class="page-break"></div>

---

## 2. Hot-Swapping Clients

Use `set_client()` to switch to a different provider mid-conversation. The
new client starts with a **fresh conversation history** (no messages carried
over from the previous client), but FFAI-level history is preserved across
the swap.

In [4]:
print(f"Before swap: client={ffai.client.model}, history={len(ffai.get_client_conversation_history())} msgs")
print(f"FFAI ordered history: {len(ffai.get_all_interactions())} interactions")

ffai.set_client(openai_client)

print(f"\nAfter swap:  client={ffai.client.model}, history={len(ffai.get_client_conversation_history())} msgs")
print(f"FFAI ordered history: {len(ffai.get_all_interactions())} interactions (preserved)")

Before swap: client=mistral-small-latest, history=4 msgs
FFAI ordered history: 2 interactions

After swap:  client=gpt-4o-mini, history=0 msgs
FFAI ordered history: 2 interactions (preserved)


In [5]:
r3 = ffai.generate_response(
    prompt="What is my name? Answer honestly.",
    prompt_name="recall_openai",
)
print(f"OpenAI response (no conversation context): {r3.response}")
print(f"\nClient: {ffai.client.model}")
print(f"Client history: {len(ffai.get_client_conversation_history())} messages")

OpenAI response (no conversation context): I'm sorry, but I don't have access to personal information about you, including your name. How can I assist you today?

Client: gpt-4o-mini
Client history: 2 messages


In [6]:
import polars as pl

ordered_df = ffai.ordered_history_to_dataframe()
print("=== FFAI Ordered History (spans both clients) ===")
print(ordered_df.select([
    pl.col("sequence_number"),
    pl.col("prompt_name"),
    pl.col("model"),
    pl.col("response").str.slice(0, 60).alias("response_preview"),
]))

=== FFAI Ordered History (spans both clients) ===
shape: (3, 4)
┌─────────────────┬───────────────┬──────────────────────┬─────────────────────────────────┐
│ sequence_number ┆ prompt_name   ┆ model                ┆ response_preview                │
│ ---             ┆ ---           ┆ ---                  ┆ ---                             │
│ i64             ┆ str           ┆ str                  ┆ str                             │
╞═════════════════╪═══════════════╪══════════════════════╪═════════════════════════════════╡
│ 1               ┆ intro         ┆ mistral-small-latest ┆ Got it, Alice! I'll remember y… │
│ 2               ┆ recall        ┆ mistral-small-latest ┆ Your name is Alice!             │
│ 3               ┆ recall_openai ┆ gpt-4o-mini          ┆ I'm sorry, but I don't have ac… │
└─────────────────┴───────────────┴──────────────────────┴─────────────────────────────────┘


<div class="page-break"></div>

---

## 3. Clearing Conversation

`clear_conversation()` resets the client's message list but **does not**
affect FFAI-level history. The model loses all prior context.

In [7]:
ffai.set_client(mistral_client)
r4 = ffai.generate_response(
    prompt="My favorite color is blue.",
    prompt_name="color",
)
print(f"Before clear: {r4.response}")
print(f"Client history: {len(ffai.get_client_conversation_history())} messages")

ffai.clear_conversation()

print(f"\nAfter clear: client history={len(ffai.get_client_conversation_history())} messages")
print(f"FFAI ordered history: {len(ffai.get_all_interactions())} interactions (preserved)")

Before clear: That's a lovely choice, Alice! Blue is a calming and versatile color. Is there anything else you'd like to share or ask? 😊
Client history: 6 messages

After clear: client history=0 messages
FFAI ordered history: 4 interactions (preserved)


In [8]:
r5 = ffai.generate_response(
    prompt="What is my favorite color?",
    prompt_name="recall_after_clear",
)
print(f"After clear, model says: {r5.response}")
print("\n(The model has no memory of the conversation before clear_conversation)")

After clear, model says: I don't have access to personal information about you unless you've shared it with me in our conversation. If you'd like to tell me your favorite color, I'd be happy to remember it for future interactions! 😊

(The model has no memory of the conversation before clear_conversation)


<div class="page-break"></div>

---

## 4. Injecting Messages

Use `add_client_message()` to inject messages directly into the client's
conversation history. This lets you seed context without making an LLM call.

In [9]:
ffai.clear_conversation()

ffai.add_client_message("user", "My cat's name is Whiskers.")
ffai.add_client_message("assistant", "Got it! Whiskers is your cat.")

print(f"Injected 2 messages. Client history: {len(ffai.get_client_conversation_history())} messages")
for msg in ffai.get_client_conversation_history():
    print(f"  [{msg['role']:9s}] {msg['content']}")

Injected 2 messages. Client history: 2 messages
  [user     ] My cat's name is Whiskers.
  [assistant] Got it! Whiskers is your cat.


In [10]:
r6 = ffai.generate_response(
    prompt="What is my cat's name?",
    prompt_name="recall_injected",
)
print(f"Model recalls injected context: {r6.response}")

Model recalls injected context: Your cat's name is Whiskers!


In [11]:
print("Full client history after inject + generate:")
for msg in ffai.get_client_conversation_history():
    role = msg.get("role", "?")
    content = msg.get("content", "")
    preview = content[:70] if isinstance(content, str) else str(content)[:70]
    print(f"  [{role:9s}] {preview}")

Full client history after inject + generate:
  [user     ] My cat's name is Whiskers.
  [assistant] Got it! Whiskers is your cat.
  [user     ] What is my cat's name?
  [assistant] Your cat's name is Whiskers!


<div class="page-break"></div>

---

## 5. Cloning Clients

`client.clone()` creates a new client instance with the same configuration
but an **empty conversation history**. This is useful for parallel branches
of conversation that share a model config but shouldn't interfere with
each other.

In [12]:
branch_client = mistral_client.clone()

print(f"Original model: {mistral_client.model}")
print(f"Clone model:    {branch_client.model}")
print(f"Original history: {len(mistral_client.get_conversation_history())} messages")
print(f"Clone history:    {len(branch_client.get_conversation_history())} messages")

Original model: mistral-small-latest
Clone model:    mistral-small-latest
Original history: 4 messages
Clone history:    0 messages


In [13]:
ffai_branch = FFAI(client=branch_client)

r_branch = ffai_branch.generate_response(
    prompt="I only exist in this branch. My name is Bob.",
    prompt_name="branch_intro",
)
print(f"Branch response: {r_branch.response}")
print(f"\nBranch client history: {len(ffai_branch.get_client_conversation_history())} messages")
print(f"Original client history: {len(ffai.get_client_conversation_history())} messages")
print("\n(Branch is completely independent from the original)")

Branch response: Hello, Bob! It's nice to meet you. How can I assist you today?

Branch client history: 2 messages
Original client history: 4 messages

(Branch is completely independent from the original)


<div class="page-break"></div>

---

## 6. Client History vs FFAI History

FFAI maintains **two separate layers** of history:

- **Client history**: The raw `{role, content}` message list sent to the LLM.
  Cleared by `clear_conversation()`, reset by `set_client()`.
- **FFAI history**: Named, sequenced, timestamped interaction records.
  Never cleared by `clear_conversation()`. Persists across client swaps.

In [14]:
print("=== Client History (only current client's messages) ===")
client_hist = ffai.get_client_conversation_history()
print(f"{len(client_hist)} messages")

print("\n=== FFAI Ordered History (all interactions across all clients) ===")
ordered = ffai.ordered_history_to_dataframe()
print(ordered.select([
    pl.col("sequence_number"),
    pl.col("prompt_name"),
    pl.col("model"),
    pl.col("response").str.slice(0, 50).alias("response_preview"),
]))

=== Client History (only current client's messages) ===
4 messages

=== FFAI Ordered History (all interactions across all clients) ===
shape: (6, 4)
┌─────────────────┬────────────────────┬──────────────────────┬─────────────────────────────────┐
│ sequence_number ┆ prompt_name        ┆ model                ┆ response_preview                │
│ ---             ┆ ---                ┆ ---                  ┆ ---                             │
│ i64             ┆ str                ┆ str                  ┆ str                             │
╞═════════════════╪════════════════════╪══════════════════════╪═════════════════════════════════╡
│ 1               ┆ intro              ┆ mistral-small-latest ┆ Got it, Alice! I'll remember y… │
│ 2               ┆ recall             ┆ mistral-small-latest ┆ Your name is Alice!             │
│ 3               ┆ recall_openai      ┆ gpt-4o-mini          ┆ I'm sorry, but I don't have ac… │
│ 4               ┆ color              ┆ mistral-small-latest ┆ Tha

In [15]:
model_stats = ffai.get_model_stats_df()
print("=== Model Usage (interactions per model across entire session) ===")
print(model_stats)

prompt_stats = ffai.get_prompt_name_stats_df()
print("\n=== Prompt Name Usage ===")
print(prompt_stats)

=== Model Usage (interactions per model across entire session) ===
shape: (2, 2)
┌──────────────────────┬───────┐
│ model                ┆ count │
│ ---                  ┆ ---   │
│ str                  ┆ i64   │
╞══════════════════════╪═══════╡
│ mistral-small-latest ┆ 5     │
│ gpt-4o-mini          ┆ 1     │
└──────────────────────┴───────┘

=== Prompt Name Usage ===
shape: (6, 2)
┌────────────────────┬───────┐
│ prompt_name        ┆ count │
│ ---                ┆ ---   │
│ str                ┆ i64   │
╞════════════════════╪═══════╡
│ intro              ┆ 1     │
│ recall             ┆ 1     │
│ recall_openai      ┆ 1     │
│ color              ┆ 1     │
│ recall_after_clear ┆ 1     │
│ recall_injected    ┆ 1     │
└────────────────────┴───────┘


In [16]:
results = [r1, r2, r3, r4, r5, r6]
total_tokens = sum(r.usage.total_tokens for r in results if r.usage)
total_cost = sum(r.cost_usd for r in results)
print("=== Session Summary ===")
print(f"Total interactions: {len(results)}")
print(f"Total tokens:       {total_tokens}")
print(f"Total cost:         ${total_cost:.6f}")
print(f"Models used:        {model_stats['model'].to_list()}")
print("Clients used:       mistral, openai, clone")

=== Session Summary ===
Total interactions: 6
Total tokens:       405
Total cost:         $0.000054
Models used:        ['mistral-small-latest', 'gpt-4o-mini']
Clients used:       mistral, openai, clone


<div class="page-break"></div>

---

## Key Takeaways

| Operation | Method | Client History | FFAI History |
|---|---|---|---|
| Generate response | `generate_response()` | Appends user + assistant messages | Records interaction |
| Swap client | `set_client(client)` | **Reset** (fresh client) | **Preserved** |
| Clear conversation | `clear_conversation()` | **Cleared** | **Preserved** |
| Inject message | `add_client_message(role, content)` | Appends to client | No effect |
| Clone client | `client.clone()` | New empty history | No effect |
| Read client history | `get_client_conversation_history()` | Returns message list | N/A |
| Set client history | `set_client_conversation_history(list)` | Replaces message list | No effect |

**When to use what:**
- Use `clear_conversation()` to start a fresh topic without losing analytics
- Use `set_client()` to switch providers while keeping a single analytics stream
- Use `add_client_message()` to inject context (e.g., system setup, few-shot examples)
- Use `clone()` to run parallel conversation branches independently